In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display
from src.utils import build_data_from_suffix
from src.feats import *

In [2]:
DATA = build_data_from_suffix("morph", csv_dir="csvs")

In [3]:
out, split_results = build_result_table(DATA, filter=True)

In [19]:
# new
out.set_index("sig").sort_values("n_new", ascending=False).head(10)

,n_full,pct_full,n_old,pct_old,n_new,pct_new,total
sig,,,,,,,
"((Animacy, ОД, Anim, Inan),)",64,0.8,26,0.4,47,2.4,137
"((UPOS, S, PROPN, NOUN),)",45,0.9,37,0.9,20,1.8,102
"((UPOS, V, AUX, VERB),)",55,4.2,32,3.6,9,2.2,96
"((Voice, СТРАД, Pass, Mid),)",23,1.1,26,1.5,8,1.8,57
"((UPOS, S, PART, PRON),)",11,12.6,7,12.1,4,13.8,22
"((Tense, НЕПРОШ, Fut, Pres),)",55,7.5,8,1.5,4,1.8,67
"((UPOS, ADV, NUM, ADV),)",23,29.5,4,9.5,3,8.3,30
"((UPOS, S, DET, PRON),)",30,2.4,13,1.5,2,0.6,45
"((UPOS, NUM, NUM, ADJ),)",45,2.0,8,0.4,1,0.3,54


In [21]:
from src.utils import save_sig_cases

tab = out.sort_values("n_new", ascending=False).reset_index(drop=True)

sig = tab.iloc[4]["sig"]
subset = save_sig_cases(split_results["new"]["tokens"], sig)

In [8]:
out_noisy, split_results_noisy = build_result_table(DATA, filter=True, mode="control")

In [20]:
out_noisy.set_index("sig").sort_values("n_new", ascending=False)

,n_full,pct_full,n_old,pct_old,n_new,pct_new,total
sig,,,,,,,
"((Animacy, ОД, Anim, Inan),)",126,1.5,72,1.2,67,3.5,265
"((Gender, СРЕД, Neut, Masc),)",138,1.0,105,1.0,39,1.1,282
"((Voice, СТРАД, Pass, Mid),)",68,3.1,48,2.8,17,3.8,133
"((UPOS, A, DET, NUM),)",33,0.8,23,0.7,17,1.4,73
"((Aspect, СОВ, Perf, ∅), (Tense, ПРОШ, Past, ∅), (UPOS, V, VERB, ADJ), (VerbForm, ПРИЧ, Part, ∅), (Voice, СТРАД, Pass, ∅))",46,3.0,32,2.7,13,3.7,91
"((UPOS, NUM, NUM, DET),)",18,0.8,6,0.3,6,1.8,30
"((Animacy, НЕОД, Inan, ∅), (Case, ИМ, Nom, ∅), (Gender, СРЕД, Neut, ∅), (Number, ЕД, Sing, ∅), (UPOS, S, PRON, SCONJ))",11,1.2,6,1.0,5,1.6,22
"((Gender, СРЕД, Neut, ∅), (Number, ЕД, Sing, ∅), (UPOS, A, ADJ, ADV), (Variant, КР, Short, ∅))",11,1.9,9,2.2,4,2.3,24
"((Animacy, ОД, Anim, ∅), (Case, ВИН, Acc, Gen))",19,2.1,14,2.1,3,1.3,36


In [14]:
tab = out_noisy.sort_values("n_new", ascending=False).reset_index(drop=True)
sig = tab.iloc[6]["sig"]
sig

(('Animacy', 'НЕОД', 'Inan', '∅'),
 ('Case', 'ИМ', 'Nom', '∅'),
 ('Gender', 'СРЕД', 'Neut', '∅'),
 ('Number', 'ЕД', 'Sing', '∅'),
 ('UPOS', 'S', 'PRON', 'SCONJ'))

In [ ]:
from src.utils import save_sig_cases

tab = out_noisy.sort_values("n_new", ascending=False).reset_index(drop=True)

sig = tab.iloc[6]["sig"]
subset = save_sig_cases(split_results_noisy["new"]["tokens"], sig)

(('Animacy', 'НЕОД', 'Inan', '∅'), ('Case', 'ИМ', 'Nom', '∅'), ('Gender', 'СРЕД', 'Neut', '∅'), ('Number', 'ЕД', 'Sing', '∅'), ('UPOS', 'S', 'PRON', 'SCONJ'))


In [19]:
from src.feats import build_comparison_df, compute_error_signatures, aggregate_signatures, add_gold_pct

cmp = build_comparison_df(DATA)

# 1. Сколько токенов с ошибкой DET→PRON в comparison?
det_pron = cmp[(cmp["upos_g_ud"] == "DET") & (cmp["upos_p_ud"] == "PRON")]
print(f"1. В comparison df: {len(det_pron)} токенов")

# 2. Как STR размечает эти токены — и ошибается ли в POS?
print(f"2. STR gold UPOS:\n{det_pron['upos_g_str'].value_counts()}")
pos_in_e_str = det_pron["E_str"].apply(lambda e: "POS" in e)
print(f"   STR ошиблась в POS: {pos_in_e_str.sum()} из {len(det_pron)}")

# 3. Попадают ли они в annotated (tokens) после compute_error_signatures?
ann = compute_error_signatures(cmp, mode="clean")
det_pron_ann = ann[(ann["upos_g_ud"] == "DET") & (ann["upos_p_ud"] == "PRON")]
print(f"3. После compute_error_signatures: {len(det_pron_ann)} токенов")

# 4. Если попали — какие у них сигнатуры?
if len(det_pron_ann) > 0:
    print(f"4. Уникальных сигнатур: {det_pron_ann['sig'].nunique()}")
    print(f"   Топ-5 сигнатур:")
    print(det_pron_ann["sig"].value_counts().head())

# 5. Проходят ли эти сигнатуры фильтр n>10 и pct>1.0?
sigs = aggregate_signatures(ann)
sigs = add_gold_pct(cmp, sigs)
det_pron_sigs = sigs[sigs["sig"].apply(
    lambda s: any(t[0] == "UPOS" and t[2] == "DET" and t[3] == "PRON" for t in s)
)]
print(f"5. В таблице сигнатур: {len(det_pron_sigs)} строк")
print(det_pron_sigs[["sig", "n", "n_gold", "pct"]])

1. В comparison df: 67 токенов
2. STR gold UPOS:
upos_g_str
A    37
S    30
Name: count, dtype: int64
   STR ошиблась в POS: 25 из 67
3. После compute_error_signatures: 42 токенов
4. Уникальных сигнатур: 2
   Топ-5 сигнатур:
sig
((UPOS, S, DET, PRON),)    30
((UPOS, A, DET, PRON),)    12
Name: count, dtype: int64
5. В таблице сигнатур: 2 строк
                        sig   n  n_gold  pct
18  ((UPOS, S, DET, PRON),)  30    1239  2.4
31  ((UPOS, A, DET, PRON),)  12    4278  0.3
